In [2]:
import math
import numpy as np
import pandas as pd


K_B = 8.617333262145e-5    # Boltzmann Constant


# Voltage effect
def voltage_factor(voltage):

    return 1 / (
        1 + 0.15 * (voltage - 3.3)
    )


# Humidity effect
def humidity_factor(humidity):

    return 1 / (
        1 + 0.03 * humidity
    )


def generate_reliability_data():

    # Random number generator
    rng = np.random.default_rng(2026)

    # Batch names
    batches = [
        "B1",
        "B2",
        "B3",
        "B4"
    ]

    # Batch variation
    batch_effect = {}

    for batch in batches:

        batch_effect[batch] = (rng.normal(1.0, 0.08))

    # Test conditions
    tests = {

        "HTOL": {

            "temperatures": [100, 125, 150],
            "voltages": [4, 4.5, 5.0],         
            "humidity": [40],   
            "test_end": 2000,
            "beta": 2.2
        },

        "THB": {
            "temperatures": [85], 
            "voltages":[3.3],
            "humidity":[40, 60, 80],
            "test_end": 1500,
            "beta": 1.9
        },

        "TC": {
            "temperatures": [-40, 125],
            "voltages": [0.0],
            "humidity": [40],
            "test_end": 1200,
            "beta":1.8       
        }
    }

    # Reference values
    activation_energy = 0.45

    reference_temperature = 398.15

    reference_eta = 2000

    # Final data storage
    all_rows = []

    device_number = 1

    # Loop through all tests
    for test_name, test_info in tests.items():

        beta = test_info["beta"]

        for temperature_c in test_info["temperatures"]:

            temperature_k = (temperature_c + 273.15)

            # Arrhenius acceleration factor
            acceleration_factor = math.exp(
                (activation_energy / K_B)*
                ((1 / temperature_k) - (1 / reference_temperature))
            )

            for humidity in test_info["humidity"]:
                for voltage in test_info["voltages"]:

                    if test_name == "TC":
                        acceleration_factor = 1.0
                        if voltage == 0:
                            voltage_multiplier = 1.0
                        else:
                            voltage_multiplier = voltage_factor(voltage)
                        base_eta = (
                            reference_eta * acceleration_factor *
                            voltage_multiplier * humidity_factor(humidity)
                        )
                    else:
                        base_eta = (
                            reference_eta * acceleration_factor *
                            voltage_factor(voltage) * humidity_factor(humidity)
                        )

                    # Create devices
                    for i in range(800):
                        batch = rng.choice(batches)

                        actual_eta = ( base_eta * batch_effect[batch]
                        * rng.normal(1.0, 0.06))
                        random_probability = (rng.random())

                        failure_time = (actual_eta * (-np.log( 1 - random_probability)
                            )**(1 / beta))

                        test_end_time = (
                            test_info["test_end"]
                        )

                        censored = int(failure_time >= test_end_time
                        )

                        row = {

                            "Device_ID":
                            f"D{device_number:04d}",

                            "Test_Type":
                            test_name,

                            "Stress_Temperature_C":
                            temperature_c,

                            "Stress_Voltage_V":
                            voltage,

                            "Humidity_Percent":
                            humidity,

                            "Failure_Time_Hours":
                            min(
                                failure_time,
                                test_end_time
                            ),

                            "Censored":
                            censored,

                            "Batch_ID":
                            batch
                        }

                        all_rows.append(
                            row
                        )

                        device_number += 1

    return pd.DataFrame(
        all_rows
    )

In [4]:
df = generate_reliability_data()

df.to_csv(
    "reliability_dataset1.csv",
    index=False
)

print(
    "CSV file saved successfully."
)

CSV file saved successfully.
